# EMG-to-Pose LSTM Training

This notebook trains an LSTM model to predict hand joint angles from EMG signals.

**Steps:**
1. Setup: clone repo, install deps, download data
2. Configure training parameters
3. Load data, build model, train
4. Evaluate and visualize results

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q h5py scipy pandas matplotlib

import os

# Clone the repo (skip if already cloned or running locally)
REPO_DIR = "Erdos_dl_emg-pose"
if not os.path.exists(REPO_DIR):
    # UPDATE this URL to your actual repo
    !git clone https://github.com/YOUR_USERNAME/Erdos_dl_emg-pose.git
else:
    print(f"Repo already cloned at {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Download mini dataset (~600 MiB)
DATA_DIR = "emg2pose_dataset_mini"
if not os.path.exists(DATA_DIR):
    print("Downloading mini dataset...")
    !curl -sL "https://fb-ctrl-oss.s3.amazonaws.com/emg2pose/emg2pose_dataset_mini.tar" -o emg2pose_dataset_mini.tar
    !tar -xf emg2pose_dataset_mini.tar
    !rm emg2pose_dataset_mini.tar
    print("Done.")
else:
    print(f"Dataset already exists at {DATA_DIR}")

n_files = len([f for f in os.listdir(DATA_DIR) if f.endswith('.hdf5')])
print(f"HDF5 files: {n_files}")

## 2. Import from repo modules

In [ ]:
import torch
from pathlib import Path

# Import from the repo's .py files
from load_data import get_dataloaders, BATCH_SIZE
from model import EMGPoseLSTM, SequentialEMGPoseLSTM
from train import (
    train_one_epoch, evaluate,
    save_history, save_model, plot_losses,
)

print("All modules imported successfully.")

## 3. Configuration

In [ ]:
CONFIG = {
    # Data
    "data_dir": "emg2pose_dataset_mini",   # Path to HDF5 files
    "test_mode": False,                     # True = synthetic data (no download)
    "use_val": False,                       # True = include validation set

    # Model
    "hidden_size": 256,
    "num_layers": 2,

    # Training
    "epochs": 10,
    "lr": 1e-3,
    "batch_size": 64,
    "patience": 20,

    # Output
    "output_dir": "checkpoints",
}

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

## 4. Load Data & Build Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

# --- Data ---
loaders = get_dataloaders(
    data_dir=CONFIG["data_dir"],
    test_mode=CONFIG["test_mode"],
    batch_size=CONFIG["batch_size"],
    use_val=CONFIG["use_val"],
)

print(f"\n{'Split':<8} {'Batches':>8} {'Samples':>10}")
for name, loader in loaders.items():
    print(f"{name:<8} {len(loader):>8} {len(loader.dataset):>10}")

# --- Inspect a batch ---
batch = next(iter(loaders["train"]))
print(f"\nBatch shapes:")
for key, val in batch.items():
    if isinstance(val, torch.Tensor):
        print(f"  {key:<20} {str(val.shape):<25} dtype={val.dtype}")

In [ ]:
# --- Model ---
model = EMGPoseLSTM(
    hidden_size=CONFIG["hidden_size"],
    num_layers=CONFIG["num_layers"],
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: EMGPoseLSTM ({n_params:,} parameters)")
print(f"  hidden_size={CONFIG['hidden_size']}, num_layers={CONFIG['num_layers']}")

# --- Optimizer ---
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])

# --- Output dir ---
output_dir = Path(CONFIG["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

## 5. Train

In [ ]:
best_loss = float("inf")
epochs_without_improvement = 0
use_val = CONFIG["use_val"] and "val" in loaders
patience = CONFIG["patience"]
max_epochs = CONFIG["epochs"]

history = {"train_loss": [], "test_mae": []}
if use_val:
    history["val_loss"] = []

print(f"Training for up to {max_epochs} epochs (patience={patience})")
print(f"  Checkpoint metric: {'val_loss' if use_val else 'test_mae'}\n")

for epoch in range(1, max_epochs + 1):
    train_loss = train_one_epoch(model, loaders["train"], optimizer, device)
    test_mae = evaluate(model, loaders["test"], device)

    history["train_loss"].append(train_loss)
    history["test_mae"].append(test_mae)

    if use_val:
        val_loss = evaluate(model, loaders["val"], device)
        history["val_loss"].append(val_loss)
        checkpoint_metric = val_loss
    else:
        checkpoint_metric = test_mae

    improved = ""
    if checkpoint_metric < best_loss:
        best_loss = checkpoint_metric
        epochs_without_improvement = 0
        save_model(model, optimizer, epoch, best_loss,
                   save_path=output_dir / "best_model.pt")
        improved = " *"
    else:
        epochs_without_improvement += 1

    log = f"Epoch {epoch:3d}/{max_epochs}  train_loss={train_loss:.4f}"
    if use_val:
        log += f"  val_loss={val_loss:.4f}"
    log += f"  test_mae={test_mae:.4f}{improved}"
    print(log)

    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping at epoch {epoch}")
        break

# Save history
save_history(history, save_path=output_dir / "loss_history.json")

## 6. Visualize & Evaluate

In [ ]:
# Plot training curves (displays inline in Colab)
plot_losses(history, save_path=output_dir / "training_curves.png")

In [ ]:
# Load best model and evaluate on test set
metric_name = "val_loss" if use_val else "test_mae"
print(f"Best {metric_name}: {best_loss:.4f}")

best_ckpt = output_dir / "best_model.pt"
if best_ckpt.exists():
    checkpoint = torch.load(best_ckpt, map_location=device, weights_only=True)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded best model from epoch {checkpoint['epoch']}")

test_loss = evaluate(model, loaders["test"], device)
print(f"Test loss (MAE): {test_loss:.4f}")

In [ ]:
# Download outputs from Colab (uncomment to use)

# from google.colab import files
# files.download(str(output_dir / "best_model.pt"))
# files.download(str(output_dir / "loss_history.json"))
# files.download(str(output_dir / "training_curves.png"))